# 21 — Agents, Planning, Memory, and Orchestration

**Network LLM Engineering — Part V — Agentic Systems**

### Learning goals
- Understand what makes a system agentic
- Build a bounded troubleshooting state machine
- Distinguish memory from context and model weights

## What is an agent?

An agent is not merely "an LLM with a long prompt".
A useful operational definition is a bounded loop with:

`state -> model/decision -> action/tool -> observation -> state update -> termination`

Production agents also need:
- budgets/timeouts,
- allowed actions,
- retries,
- validation,
- audit trails,
- human escalation.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class IncidentState:
    problem: str
    evidence: list = field(default_factory=list)
    hypotheses: list = field(default_factory=list)
    checks_done: list = field(default_factory=list)
    status: str = "investigating"

s = IncidentState("R1 Gi0/1 reports errors")
s.evidence.append("Gi0/1 operational up")
s.evidence.append("CRC counter = 432 and increasing")
s.hypotheses.append("physical/transceiver/cabling issue")
print(s)

## Memory vocabulary

- **context:** tokens available to the current model call.
- **short-term memory/state:** structured data retained during the task.
- **long-term memory:** persisted database content across tasks.
- **weights:** learned model parameters.

These are four different mechanisms.

In [ ]:
def next_step(state):
    if not state.evidence:
        return "collect basic interface state"
    if any("CRC" in x for x in state.evidence) and "peer counters" not in state.checks_done:
        return "collect peer counters and optics"
    return "summarize and escalate if evidence remains insufficient"

print(next_step(s))

## Single agent vs multiple agents

More agents do not automatically mean more intelligence.
Multi-agent designs add coordination cost, latency, failure modes, and evaluation complexity.

Use specialization when boundaries are real: e.g., routing diagnosis, security investigation, change verification.
Do not split one simple task into five agents merely because the architecture diagram looks sophisticated.

### Exercise

Define termination conditions for a troubleshooting agent:
- evidence sufficient,
- max tool calls,
- timeout,
- action requires approval,
- conflicting evidence,
- tool unavailable.